In [1]:
import pandas as pd
import numpy as np

In [2]:
df_fi_hard = pd.read_parquet("dane/interim/fact_inka_hard_records_2023_2026.parquet")

In [3]:
raws , cols = df_fi_hard.shape
print(f"Rekordów: {raws:,}\nKolumn: {cols}")

Rekordów: 4,106,901
Kolumn: 30


In [4]:
# Konwersja typów przed concat - pandas nullable types
# int64 nie obsługuje NaN - po concat z dni_zerowe (gdzie kolumny mają NaN)
# pandas konwertuje int64 → float64 i bool → object
# Int64/boolean (nullable) zachowują typ nawet przy NaN

cols_int64_to_Int64 = ['Kolejnosc', 'NrPozycji', 'TypPoz', 'KolejnyWDniu', 'AsId', 'Mnoznik']
for col in cols_int64_to_Int64:
    df_fi_hard[col] = df_fi_hard[col].astype('Int64')

# bool → boolean (nullable)
df_fi_hard['WplywNaStan'] = df_fi_hard['WplywNaStan'].astype('boolean')
df_fi_hard['CzyNiechciane'] = df_fi_hard['CzyNiechciane'].astype('boolean')

In [5]:
# odfiltrowanie samej sprzedazy

df_fi_hard_21 = df_fi_hard[
    (df_fi_hard['TypDok'] == 21)
].copy()
print(f"Rekordów sprzedaży: {len(df_fi_hard_21):,}")
print(f"Unikalnych TowId: {df_fi_hard_21['TowId'].nunique():,}")

Rekordów sprzedaży: 3,373,975
Unikalnych TowId: 12,481


In [6]:
# Daty start/koniec per SKU z df_fi_hard_21
data_max_global = df_fi_hard_21['Data'].max()

daty_sku = (
    df_fi_hard_21
    .groupby('TowId')['Data']
    .agg(DataStart='min', DataKoniec='max')
    .reset_index()
)

daty_sku['DataKoniec'] = (
    (daty_sku['DataKoniec'] + pd.Timedelta(days=14))
    .clip(upper=data_max_global)
)

# Budowa szkieletu
kalendarze = []
for _, row in daty_sku.iterrows():
    dni = pd.date_range(start=row['DataStart'], end=row['DataKoniec'], freq='D')
    kalendarze.append(pd.DataFrame({'TowId': row['TowId'], 'Data': dni}))

kalendarz = pd.concat(kalendarze, ignore_index=True)
print(f"Szkielet kalendarza: {len(kalendarz):,} rekordów, {kalendarz['TowId'].nunique()} SKU")

Szkielet kalendarza: 7,058,441 rekordów, 12481 SKU


In [7]:
# concat z df_fi_hard_21

# Dni które mają transakcje
dni_ze_sprzedaza = df_fi_hard_21[['TowId', 'Data']].drop_duplicates()

# Dni zerowe — brakujące w sprzedaży
dni_zerowe = kalendarz.merge(
    dni_ze_sprzedaza, on=['TowId', 'Data'], how='left', indicator=True
)
dni_zerowe = dni_zerowe[dni_zerowe['_merge'] == 'left_only'].drop(columns='_merge')

# Wypełnienie podstawowych pól
dni_zerowe['TypDok']         = 21
dni_zerowe['DokId']          = -1
dni_zerowe['IloscPlus']      = 0
dni_zerowe['IloscMinus']     = 0
dni_zerowe['Wartosc']        = 0
dni_zerowe['AktywnyDok']     = 1
dni_zerowe['AktywnyTow']     = 1
dni_zerowe['Dokument']       = 'DF'
dni_zerowe['WplywNaStan']    = True
dni_zerowe['MetodaLiczenia'] = 'IP'
dni_zerowe['Mnoznik']        = -1
dni_zerowe['TypRuchu']       = 'sprzedaz'
dni_zerowe['CzyNiechciane']  = False
dni_zerowe['TypPoz']         = 4
dni_zerowe['NrDok']        = 'BRAK'
dni_zerowe['KolejnyWDniu'] = -1
dni_zerowe['Razem']        = 0
dni_zerowe['DoZaplaty']    = 0
dni_zerowe['Zaplacono']    = 0
dni_zerowe['Kolejnosc']    = -1
dni_zerowe['NrPozycji']    = -1
dni_zerowe['CenaPoRab']    = np.nan  # ffill później

# Concat z df_fi_hard_21
kalendarz_full = pd.concat([df_fi_hard_21, dni_zerowe], ignore_index=True)

print(f"Shape: {len(kalendarz_full):,} rekordów")
print(f"Dni zerowych: {len(dni_zerowe):,}")

Shape: 8,802,515 rekordów
Dni zerowych: 5,428,540


In [8]:
kalendarz_full = kalendarz_full.sort_values(['TowId', 'Data'])
kalendarz_full['CenaPoRab'] = (
    kalendarz_full.groupby('TowId')['CenaPoRab'].ffill()
)
print(f"NaN w CenaPoRab po ffill: {kalendarz_full['CenaPoRab'].isna().sum():,}")

NaN w CenaPoRab po ffill: 0


In [9]:
# Merge kolumn stałych per SKU
sku_info = (
    df_fi_hard[['TowId', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'NazwaAsort']]
    .drop_duplicates('TowId')
)

# Usuń stare kolumny z NaN i dołącz poprawne
kalendarz_full = kalendarz_full.drop(columns=['AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'NazwaAsort'])
kalendarz_full = kalendarz_full.merge(sku_info, on='TowId', how='left')

print(f"Shape: {kalendarz_full.shape}")
print(f"NaN w NazwaTow: {kalendarz_full['NazwaTow'].isna().sum()}")
print(f"NaN w AsId: {kalendarz_full['AsId'].isna().sum()}")

Shape: (8802515, 30)
NaN w NazwaTow: 0
NaN w AsId: 0


In [13]:
print(f"Shape: {kalendarz_full.shape}")
print(f"Kolumny: {kalendarz_full.columns.tolist()}")
print(f"NaN per kolumna:\n{kalendarz_full.isna().sum()}")
print(f"\nUnikalnych SKU: {kalendarz_full['TowId'].nunique()}")
print(f"Zakres dat: {kalendarz_full['Data'].min()} → {kalendarz_full['Data'].max()}")

Shape: (9535441, 30)
Kolumny: ['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'Data', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AktywnyTow', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'NazwaAsort']
NaN per kolumna:
DokId                   0
Kolejnosc               0
NrPozycji               0
TowId                   0
TypPoz                  0
IloscPlus               0
IloscMinus              0
CenaPoRab               0
Wartosc                 0
Data                    0
KolejnyWDniu            0
NrDok                   0
TypDok                  0
AktywnyDok              0
Razem                   0
DoZaplaty               0
Zaplacono               0
AktywnyTow              0
Dokument                0
WplywNaStan             0
MetodaLiczenia          0
Mnoznik                 0
TypRuchu             

In [11]:
df_fi_hard_rest = df_fi_hard[df_fi_hard['TypDok'] != 21]
kalendarz_full = pd.concat([kalendarz_full, df_fi_hard_rest], ignore_index=True)

C:\Users\huber\AppData\Local\Temp\ipykernel_9104\2693979635.py:2: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  kalendarz_full = pd.concat([kalendarz_full, df_fi_hard_rest], ignore_index=True)


In [12]:
kalendarz_full.to_parquet(
    "dane/interim/kalendarz_full.parquet",
    engine='pyarrow',
    compression='zstd',
    index=False
)